# Untold Lores — Combined TTS Server
Serves two TTS engines on a single ngrok URL:
- `POST /tts` → Khmer TTS (edge-tts, `km-KH-PisethNeural`)
- `POST /f5tts` → English voice cloning (F5-TTS, clones your ElevenLabs voice)

**Every session:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run Cell 1 → install dependencies
3. Run Cell 2 → load F5-TTS model
4. Run Cell 3 → upload reference audio (takes ~5 seconds)
5. Run Cell 4 → start server
6. Run Cell 5 → expose with ngrok, copy URL to `.env`
7. Run Cell 6 → keep alive

In [ ]:
# Cell 1 — Install dependencies
!pip install -q git+https://github.com/SWivid/F5-TTS.git edge-tts flask pyngrok nest_asyncio
!apt-get install -qq ffmpeg

In [ ]:
# Cell 2 — Load F5-TTS model (downloads weights on first run ~2 min, cached after)
from f5_tts.api import F5TTS

print('🔧 Loading F5-TTS model...')
tts = F5TTS()
print('✅ F5-TTS ready!')

In [ ]:
# Cell 3 — Upload reference audio and pre-transcribe (run every session)
from google.colab import files

print('Upload your ElevenLabs reference audio (.mp3 or .wav)...')
uploaded = files.upload()
reference_file = list(uploaded.keys())[0]
print(f'✅ Reference file ready: {reference_file}')

print('Transcribing reference audio (one-time, ~10 seconds)...')
reference_text = tts.transcribe(reference_file)
print(f'✅ Reference text: "{reference_text}"')

In [ ]:
# Cell 4 — Start combined Flask server
import edge_tts
import asyncio, io, os, subprocess, tempfile, threading
import nest_asyncio
from flask import Flask, request, send_file, jsonify

nest_asyncio.apply()

app = Flask(__name__)
TMP_DIR = tempfile.mkdtemp()

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'reference_file': reference_file})

# ── Khmer TTS (edge-tts) ──────────────────────────────────────────────────────
@app.route('/tts', methods=['POST'])
def khmer_tts():
    data  = request.get_json()
    text  = data.get('text', '')
    voice = data.get('voice', 'km-KH-PisethNeural')
    rate  = data.get('rate', '-10%')
    pitch = data.get('pitch', '-5Hz')

    if not text:
        return jsonify({'error': 'text is required'}), 400

    async def generate():
        communicate = edge_tts.Communicate(text, voice=voice, rate=rate, pitch=pitch)
        buf = io.BytesIO()
        async for chunk in communicate.stream():
            if chunk['type'] == 'audio':
                buf.write(chunk['data'])
        buf.seek(0)
        return buf

    buf = asyncio.get_event_loop().run_until_complete(generate())
    return send_file(buf, mimetype='audio/mpeg', as_attachment=True, download_name='audio.mp3')

# ── English voice cloning (F5-TTS) ───────────────────────────────────────────
@app.route('/f5tts', methods=['POST'])
def f5tts():
    data = request.get_json()
    text = data.get('text', '')

    if not text:
        return jsonify({'error': 'text is required'}), 400

    wav_path = os.path.join(TMP_DIR, f'f5_{os.getpid()}.wav')
    mp3_path = wav_path.replace('.wav', '.mp3')

    try:
        tts.infer(
            ref_file=reference_file,
            ref_text=reference_text,  # pre-transcribed — no Whisper on every call
            gen_text=text,
            file_wave=wav_path,
        )
        subprocess.run(
            ['ffmpeg', '-i', wav_path, '-codec:a', 'libmp3lame', '-qscale:a', '2', '-y', mp3_path],
            check=True, capture_output=True
        )
        response = send_file(mp3_path, mimetype='audio/mpeg', as_attachment=True, download_name='audio.mp3')
        return response
    except Exception as e:
        return jsonify({'error': str(e)}), 500
    finally:
        if os.path.exists(wav_path):
            os.remove(wav_path)
        if os.path.exists(mp3_path):
            os.remove(mp3_path)

t = threading.Thread(target=lambda: app.run(port=5001, use_reloader=False, threaded=False))
t.daemon = True
t.start()
print('✅ Combined TTS server started on port 5001')
print('   /tts    → Khmer (edge-tts)')
print('   /f5tts  → English voice cloning (F5-TTS)')

In [ ]:
# Cell 5 — Expose with ngrok
from pyngrok import ngrok

NGROK_TOKEN = 'PASTE_YOUR_NGROK_TOKEN_HERE'  # <-- replace this

ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5001).public_url

print('=' * 60)
print('🚀 Combined TTS server is live!')
print('')
print('Add both lines to your .env file (same URL):')
print(f'KHMER_TTS_NGROK_URL={public_url}')
print(f'F5TTS_NGROK_URL={public_url}')
print('=' * 60)

In [ ]:
# Cell 6 — Keep session alive (run this, leave it running)
import time
print('⏳ Keeping session alive... (interrupt kernel when done generating)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)